# Framework certification runner
Repository-owned safe starting point. Bootstrap preparation is not certification PASS. Live mutation authorizations remain false in this notebook.

In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

root = Path('/lakehouse/default/Files/framework_cert')
candidate_path = root / 'CANDIDATE.json'
candidate = json.loads(candidate_path.read_text(encoding='utf-8'))
wheel = root / candidate['wheel_filename']
digest = hashlib.sha256(wheel.read_bytes()).hexdigest()
assert digest == candidate['wheel_sha256'], 'staged Framework wheel SHA256 mismatch'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', str(wheel)])
print('Exact Framework wheel verified and installed:', candidate['candidate_git_sha'])


In [ ]:
os.environ['FABRIC_SQL_AUTH_MODE'] = 'fabric-user'
os.environ['CONTROL_PLANE_SQL_SERVER'] = '__CONTROL_PLANE_SERVER__'
os.environ['CONTROL_PLANE_SQL_DATABASE'] = '__CONTROL_PLANE_DATABASE__'
os.environ['WAREHOUSE_SQL_SERVER'] = '__WAREHOUSE_SERVER__'
os.environ['WAREHOUSE_SQL_DATABASE'] = '__WAREHOUSE_DATABASE__'
print('Fabric-user SQL runtime bindings prepared; no token printed')


In [ ]:
from fabric_data_framework.certification import certify, print_certification_summary

report = certify(
    spark=spark,
    candidate_manifest_path=candidate_path,
    wheel_path=wheel,
    output_dir=root / 'certification-output',
    environment='__CERTIFICATION_ENVIRONMENT__',
    customer_inputs_root=root / 'customer-inputs',
    allow_control_plane_migration=False,
    allow_control_plane_writes=False,
    allow_pipeline_execution=False,
    allow_capture_execution=False,
    allow_warehouse_execution=False,
    allow_warehouse_fault_injection=False,
    allow_warehouse_session_termination=False,
    allow_business_path_execution=False,
    allow_scenario_mutation=False,
)
print_certification_summary(report)
